In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher
from tqdm import tqdm

In [ ]:
TRAIN_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"
 


LABELS            = ["A", "B", "C", "D", "E"]
RETRIEVAL_THRESHOLD = 0.75   # min similarity to use retrieval
TOP_K             = 10        # top-K train matches to combine

def full_text(row):
    return str(row["prompt"]) + " " + " ".join(str(row[c]) for c in LABELS)


In [ ]:
def map_train_answer_to_test(test_row, train_row):
    train_ans_idx = LABELS.index(train_row["answer"])
    test_opts     = [str(test_row[c]).strip()  for c in LABELS]
    train_opts    = [str(train_row[c]).strip() for c in LABELS]

    # Fast path: all options identical -> answer label maps directly
    if test_opts == train_opts:
        return train_ans_idx

    # Build 5x5 character-similarity matrix
    sim_mat = np.zeros((5, 5))
    for i in range(5):
        for j in range(5):
            sim_mat[i, j] = SequenceMatcher(
                None, train_opts[i].lower(), test_opts[j].lower()
            ).ratio()

    # Phase 1: exact string matches (highest confidence)
    matched  = {}   # train_idx -> test_idx
    used     = set()
    for i in range(5):
        for j in range(5):
            if j not in used and train_opts[i] == test_opts[j]:
                matched[i] = j
                used.add(j)
                break

    # Phase 2: greedy char-sim assignment for remaining options
    unmatched_train = [i for i in range(5) if i not in matched]
    unmatched_test  = [j for j in range(5) if j not in used]

    while unmatched_train and unmatched_test:
        sub_mat   = sim_mat[np.ix_(unmatched_train, unmatched_test)]
        best_flat = int(np.argmax(sub_mat))
        ri, ci    = divmod(best_flat, len(unmatched_test))
        matched[unmatched_train[ri]] = unmatched_test[ci]
        unmatched_train.pop(ri)
        unmatched_test.pop(ci)

    return matched.get(train_ans_idx, train_ans_idx)


In [ ]:
def heuristic_scores(row):
    prompt  = str(row["prompt"])
    options = [str(row[c]) for c in LABELS]
    lens    = np.array([len(o) for o in options], dtype=float)
    len_s   = lens / (lens.sum() + 1e-9)
    wcnt    = np.array([len(o.split()) for o in options], dtype=float)
    wc_s    = wcnt / (wcnt.sum() + 1e-9)
    try:
        v    = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit_transform(
                   [prompt] + options)
        ts   = cosine_similarity(v[0:1], v[1:])[0]
        tf_s = ts / (ts.sum() + 1e-9)
    except Exception:
        tf_s = np.full(5, 0.2)
    std = np.std(lens)
    if std >= 20:
        return 0.85 * len_s + 0.03 * tf_s + 0.12 * wc_s
    else:
        return 0.60 * len_s + 0.30 * tf_s + 0.10 * wc_s

In [ ]:
def map_at_3(pred_strings, true_labels):
    total = 0.0
    for pred_str, truth in zip(pred_strings, true_labels):
        for k, p in enumerate(pred_str.split()[:3]):
            if p == truth:
                total += 1.0 / (k + 1)
                break
    return total / len(true_labels) if true_labels else 0.0

In [ ]:
RUN_VALIDATION = False   # set True to run 5-fold CV on training data first

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")

train_rows = [row for _, row in train_df.iterrows()]
test_rows  = [row for _, row in test_df.iterrows()]


print("\nBuilding TF-IDF index...")
train_texts = [full_text(r) for r in train_rows]
test_texts  = [full_text(r) for r in test_rows]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=100000
)
vectorizer.fit(train_texts + test_texts)

train_mat = vectorizer.transform(train_texts)
test_mat  = vectorizer.transform(test_texts)

print("Computing similarity matrix...")
sims_matrix = cosine_similarity(test_mat, train_mat)   # (500, 2000)

In [ ]:
if RUN_VALIDATION:
    from sklearn.model_selection import KFold
    kf       = KFold(n_splits=5, shuffle=True, random_state=42)
    truths   = train_df["answer"].tolist()
    all_map3 = 0.0
    n_all    = 0
    print("\nRunning 5-fold cross-validation...")

    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(train_df)))):
        tr_texts_f = [train_texts[i] for i in tr_idx]
        va_texts_f = [train_texts[i] for i in va_idx]
        tr_rows_f  = [train_rows[i]  for i in tr_idx]

        fvec = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True,
                               max_features=100000)
        fvec.fit(tr_texts_f + va_texts_f)
        fold_sims = cosine_similarity(
            fvec.transform(va_texts_f), fvec.transform(tr_texts_f)
        )

        for vi, val_i in enumerate(va_idx):
            row_sims   = fold_sims[vi]
            top_idxs   = np.argsort(row_sims)[::-1][:TOP_K]
            opt_scores = np.zeros(5)

            if row_sims[top_idxs[0]] >= RETRIEVAL_THRESHOLD:
                for rank, j in enumerate(top_idxs):
                    sim = row_sims[j]
                    if sim < RETRIEVAL_THRESHOLD:
                        break
                    mapped = map_train_answer_to_test(train_rows[val_i], tr_rows_f[j])
                    opt_scores[mapped] += sim / (rank + 1)
            else:
                opt_scores = heuristic_scores(train_rows[val_i])

            order = np.argsort(opt_scores)[::-1]
            top3  = [LABELS[i] for i in order[:3]]
            for k, p in enumerate(top3):
                if p == truths[val_i]:
                    all_map3 += 1.0 / (k + 1)
                    break
            n_all += 1

        print(f"  Fold {fold+1}/5 done")

    print(f"Cross-validated MAP@3: {all_map3/n_all:.4f}")

In [ ]:
print("\nGenerating predictions on test set...")
predictions   = []
retrieval_cnt = 0
heuristic_cnt = 0

for ti, test_row in enumerate(tqdm(test_rows, desc="Predicting")):
    row_sims   = sims_matrix[ti]
    top_idxs   = np.argsort(row_sims)[::-1][:TOP_K]
    max_sim    = row_sims[top_idxs[0]]
    opt_scores = np.zeros(5)

    if max_sim >= RETRIEVAL_THRESHOLD:
        for rank, j in enumerate(top_idxs):
            sim = row_sims[j]
            if sim < RETRIEVAL_THRESHOLD:
                break
            mapped = map_train_answer_to_test(test_row, train_rows[j])
            opt_scores[mapped] += sim / (rank + 1)
        retrieval_cnt += 1
    else:
        opt_scores = heuristic_scores(test_row)
        heuristic_cnt += 1

    order = np.argsort(opt_scores)[::-1]
    top3  = [LABELS[i] for i in order[:3]]
    predictions.append({"ID": test_row["id"], "Prediction": " ".join(top3)})

print(f"\nRetrieval used : {retrieval_cnt}/500")
print(f"Heuristic used : {heuristic_cnt}/500")

submission = pd.DataFrame(predictions)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")
print(submission.head(10).to_string(index=False))


In [3]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [4]:
# print(train_df.shape)
# print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
# print(train_df.columns.tolist())
# print(test_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
